In [ ]:
from typing import no_type_check

import zixy.qubit.pauli as zqp
from guppylang import guppy
from guppylang.std.builtins import array
from guppylang.std.quantum import qubit

from guppyalgos.primitives.gate_decompositions.and_op import (
    temp_and_compute,
    temp_and_uncompute,
)
from guppyalgos.primitives.gate_decompositions.cnx.cnx import cnx
from guppyalgos.algorithms.block_encoding.lcu import LCU, LCUData, build_unary_iteration_select
from guppyalgos.algorithms.block_encoding.qubitization.qubitization import Qubitization
from guppyalgos.primitives.subroutines.reflection import Reflection
from guppyalgos.algorithms.state_preparation import multiplexor_prep
from guppyalgos.tests.helpers import (
    Endianness,
    assert_allclose_ignorephase,
    chebyshev_power_matrix,
    get_unitary_projected,
)

# LCU qubitization

The helper API constructs the qubitization ingredients directly from the Hermitian `zixy` Hamiltonian

$$
H = 0.4 Z_0 - 0.2 X_1 + 0.3 Y_0Y_1 + 0.1 Z_0X_1.
$$

`LCUData.from_hamiltonian` extracts the coefficients and Pauli strings, `multiplexor_prep` builds `PREPARE`, and `build_unary_iteration_select` builds `SELECT` using temporary-AND compute and uncompute operations. The qubitization circuit constructs `LCU` directly from these oracles and checks its projected second power against the second Chebyshev polynomial, $T_2(H / \lVert H \rVert_1)$.

Unary iteration allocates $n_{\mathrm{prep}} - 1$ work qubits internally. The statevector test passes this count to `get_unitary_projected` through `n_extra_qubits`, reserving enough emulator qubits without adding the work register to the circuit interface.

In [3]:
hamiltonian = zqp.RealTermSum.from_str(
    "(0.4, Z0), (-0.2, X1), (0.3, Y0 Y1), (0.1, Z0 X1)",
    2,
)
lcu_data = LCUData.from_hamiltonian(hamiltonian)

prepare = multiplexor_prep(lcu_data.amplitudes)

select = build_unary_iteration_select(
    lcu_data,
    comp_and_op=temp_and_compute,
    uncomp_and_op=temp_and_uncompute,
)

dagger = object()

@guppy
@no_type_check
def unprepare(prep: array[qubit, 2]) -> None:
    with dagger:
        prepare(prep)

In [4]:
power = 2

@guppy
@no_type_check
def qubitize(
    prep: array[qubit, 2],
    state: array[qubit, 2],
) -> None:
    lcu = LCU(prepare, select, unprepare)
    reflection = Reflection[2, 1](cnx)
    Qubitization(lcu, reflection).power(prep, state, power)

projected_qubitization = get_unitary_projected(
    qubitize,
    lcu_data.n_state_qubits,
    {"prep": [False] * lcu_data.n_prep_qubits},
    n_extra_qubits=lcu_data.n_prep_qubits - 1,
    endianness=Endianness.LITTLE,
)

normalized_hamiltonian = (
    hamiltonian.to_sparse_matrix().toarray() / lcu_data.l1_norm
)
expected_chebyshev = chebyshev_power_matrix(normalized_hamiltonian, power)
assert_allclose_ignorephase(projected_qubitization, expected_chebyshev)